<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/easy/augmentation/augmentation.ipynb)

This problem is not auto-graded: run the notebook's own checks, then compare with the solution notebook.


# Problem: Use torchvision.transforms for Data Augmentation

### Problem Statement
Data augmentation applies random, label-preserving changes to the training images every time they are loaded. The model sees a slightly different version of each image in every epoch, which acts as a regularizer. Your task is to build the standard CIFAR-10 augmentation pipeline with **torchvision.transforms**, build a separate *deterministic* pipeline for the test set, and visualize a batch of augmented images.

### Requirements
1. **Training pipeline** (`train_transform`), applied in this order:
   - **Random Horizontal Flip**: flip the image with probability 0.5.
   - **Random Crop**: pad 4 pixels on every side, then randomly crop back to 32×32 (`RandomCrop(32, padding=4)`). This shifts the image by up to ±4 pixels without changing its size, so the same model can consume train and test images.
   - **ToTensor**: PIL image with values 0–255 in H×W×C layout → float tensor in [0, 1] in C×H×W layout.
   - **Normalize**: subtract the per-channel CIFAR-10 mean and divide by the per-channel std (the values are given in the code).
2. **Test pipeline** (`test_transform`): only `ToTensor` and `Normalize`, with **no random transforms**. Augmentation is a training-time regularizer. Evaluation must be deterministic and measure the model on the real test images.
3. **Visualize** a batch of augmented training images with **matplotlib**. Undo the normalization first (`x * std + mean`) so the colors look right.

### Constraints
- Geometric transforms such as flip and crop act on the PIL image, so they come before `ToTensor`. `Normalize` works on tensors, so it comes last.
- Use the CIFAR-10 statistics (mean ≈ (0.4914, 0.4822, 0.4465), std ≈ (0.2470, 0.2435, 0.2616)), not ImageNet's.

### Notes
- `torchvision.transforms.v2` is the newer API. It has the same transform names and can also transform bounding boxes and masks together with the image. The classic `torchvision.transforms` used here still works.
- Using a random crop *size* smaller than 32 (e.g. 28×28) would change the input resolution. You would then need a matching `CenterCrop(28)` at test time and a model built for 28×28 inputs.

<details>
  <summary>💡 Hint</summary>
  Build two <code>transforms.Compose</code> pipelines: <code>[RandomHorizontalFlip, RandomCrop(32, padding=4), ToTensor, Normalize]</code> for training, and only <code>[ToTensor, Normalize]</code> for testing.
  <br>
  To display a normalized image, reverse the normalization (<code>img * std + mean</code>), clamp to [0, 1], and permute C×H×W → H×W×C for <code>plt.imshow</code>.
</details>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
# Per-channel statistics of the CIFAR-10 training set
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# TODO: training pipeline: random horizontal flip, random crop (pad 4, crop back to 32), ToTensor, Normalize
train_transform = transforms.Compose([
    ...
])

# TODO: test pipeline: deterministic, no random augmentation
test_transform = transforms.Compose([
    ...
])

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Checks: augmentation keeps the shape, is random for training, and is off for testing
img, _ = train_dataset[0]
assert img.shape == (3, 32, 32), f"expected a 3x32x32 tensor, got {tuple(img.shape)}"
draws = [train_dataset[0][0] for _ in range(8)]
assert any(not torch.equal(draws[0], d) for d in draws[1:]), "training images should be randomly augmented"
assert torch.equal(test_dataset[0][0], test_dataset[0][0]), "the test pipeline must be deterministic"
print("All checks passed")

In [ ]:
# Display a batch of augmented images
def imshow(img):
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    img = (img * std + mean).clamp(0, 1)   # undo Normalize
    plt.imshow(img.permute(1, 2, 0).numpy())  # CxHxW -> HxWxC
    plt.axis('off')
    plt.show()

# Get some random training images
images, labels = next(iter(train_loader))

# Show images
imshow(torchvision.utils.make_grid(images))